In [157]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as dataset

import numpy as np
import matplotlib.pyplot as plt

In [158]:
class CausalSelfAttention(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len) -> None:
    super().__init__()

    self.d_k = d_k
    self.n_heads = n_heads

    self.key = nn.Linear(d_model, d_k * n_heads)
    self.query = nn.Linear(d_model, d_k * n_heads)
    self.value = nn.Linear(d_model, d_k * n_heads)

    self.fc = nn.Linear(d_k * n_heads, d_model)

    cm = torch.tril(torch.ones(max_len, max_len))
    self.register_buffer(
        'causal_mask',
        cm.view(1, 1, max_len, max_len)
    )


  def forward(self, q, k, v, pad_mask=None):
    q = self.query(q)   # (N, T, d_model) -> (N, T, h*d_k)
    k = self.key(k)   # (N, T, d_model) -> (N, T, h*d_k)
    v = self.value(v)   # (N, T, d_model) -> (N, T, h*d_k)

    N = q.shape[0]
    T = q.shape[1]

    # change the shape
    # (N, T, h, d_k) -> (N, h, T, d_k)
    # in order for matrix multiply to work properly
    q = q.view(N, T, self.n_heads, self.d_k).transpose(1, 2)
    k = k.view(N, T, self.n_heads, self.d_k).transpose(1, 2)
    v = v.view(N, T, self.n_heads, self.d_k).transpose(1, 2)

    # Compute attention weights
    # (N, h, T, d_k) x (N, h, d_k, T) -> (N, h, T, T)
    attn_scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
    if pad_mask is not None:
      attn_scores = attn_scores.masked_fill(pad_mask[:, None, None, :] == 0, float('-inf'))

    attn_scores = attn_scores.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))

    attn_weights = F.softmax(attn_scores, dim=-1)

    # Compute attention weights values
    # (N, h, T, T) x (N, h, T, d_k) -> (N, h, T, d_k)
    A = attn_weights @ v

    # Reshape it back before final linear layer
    A = A.transpose(1, 2) # (N, T, h, d_k)
    A = A.contiguous().view(N, T, self.d_k * self.n_heads) # (N, T, h*d_k)

    # projection
    return self.fc(A)



In [159]:
class TransformerBlock(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, dropout_prob=0.1) -> None:
    super().__init__()

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.mha = CausalSelfAttention(d_k, d_model, n_heads, max_len)
    self.ann = nn.Sequential(
        nn.Linear(d_model, d_model * 4),
        nn.GELU(),
        nn.Linear(d_model * 4, d_model),
        nn.Dropout(dropout_prob)
    )
    self.dropout = nn.Dropout(p=dropout_prob)

  def forward(self, x, pad_mask=None):
    x = self.ln1(x + self.mha(x, x, x, pad_mask))
    x = self.ln2(x + self.ann(x))
    x = self.dropout(x)
    return x

In [160]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=2048, dropout_prob=0.1) -> None:
    super().__init__()

    self.dropout = nn.Dropout(p=dropout_prob)

    position = torch.arange(max_len).unsqueeze(1)
    exp_term = torch.arange(0, d_model, 2)
    div_term = torch.exp(exp_term * (-math.log(10000.0)/d_model))
    pe = torch.zeros(1, max_len, d_model)
    pe[0, :, 0::2] = torch.sin(position * div_term)
    pe[0, :, 1::2] = torch.cos(position * div_term)
    self.register_buffer('pe', pe)

  def forward(self, x):
    # x.shape (N, T, D)
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)

In [161]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, max_len, d_k, d_model, n_heads, n_layers, dropout_prob) -> None:
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, d_model)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)
    transformer_blocks = [
        TransformerBlock(d_k, d_model, n_heads, max_len, dropout_prob) for _ in range(n_layers)]
    self.transformer_blocks = nn.Sequential(*transformer_blocks)
    self.ln = nn.LayerNorm(d_model)
    self.fc = nn.Linear(d_model, vocab_size)

  def forward(self, x, pad_mask=None):
    x = self.embedding(x)
    x = self.pos_encoding(x)

    for block in self.transformer_blocks:
      x = block(x, pad_mask)

    # many-to-one x has shape (N x T x D) -> (N x D)
    # x = x[:, 0, :] - this is for encoder

    x = self.ln(x)
    x = self.fc(x) # many-to-many
    return x

In [162]:
model = Decoder(20_000, 1024, 16, 64, 4, 2, 0.1)

In [163]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

cuda:0


Decoder(
  (embedding): Embedding(20000, 64)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mha): CausalSelfAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
      )
      (ann): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=64, bias=True)
        (3): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (ln1): LayerNorm((64,), eps=1e-05,

In [164]:
x = np.random.randint(0, 20_000, size=(8, 512))
x_t = torch.tensor(x).to(device)

In [165]:
y = model(x_t)
y.shape

torch.Size([8, 512, 20000])

In [166]:
mask = np.ones((8, 512))
mask[:, 256:] = 0
mask_t = torch.tensor(mask).to(device)

In [167]:
y = model(x_t, mask_t)

In [168]:
y.shape

torch.Size([8, 512, 20000])

In [169]:
!pip install transformers datasets

In [170]:
from transformers import AutoTokenizer, DataCollatorWithPadding

In [171]:
checkpoint = 'distilbert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [172]:
from datasets import load_dataset

In [173]:
raw_datasets = load_dataset('glue', 'sst2')

In [174]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [175]:
raw_datasets['train'][0]

{'sentence': 'hide new secretions from the parental units ',
 'label': 0,
 'idx': 0}

In [176]:
def tokenize_fn(batch):
  return tokenizer(batch['sentence'], trancation=True)

In [177]:
tokenized_datasets = raw_datasets.map(tokenize_fn, batched=True)

In [178]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [179]:
data_collator

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='distilbert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

In [180]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1821
    })
})

In [181]:
tokenized_datasets = tokenized_datasets.remove_columns(['sentence', 'idx', 'label'])

In [182]:
tokenized_datasets = tokenized_datasets.remove_columns(['token_type_ids'])

In [183]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1821
    })
})

In [184]:
tokenized_datasets['train'][0]

{'input_ids': [101, 4750, 1207, 3318, 5266, 1121, 1103, 22467, 2338, 102],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [185]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    tokenized_datasets['train'],
    shuffle=True,
    batch_size=32,
    collate_fn=data_collator)

# valid_loader = DataLoader(
#     tokenized_datasets['validation'],
#     batch_size=32,
#     collate_fn=data_collator)

In [186]:
for batch in train_loader:
  for k, v in batch.items():
    print("k: ", k, "v.shape: ", v.shape)
  break

k:  input_ids v.shape:  torch.Size([32, 28])
k:  attention_mask v.shape:  torch.Size([32, 28])


In [187]:
tokenizer.pad_token_id

0

In [188]:
tokenizer.vocab_size

28996

In [189]:
tokenizer.model_max_length #.max_model_input_sizes[checkpoint]

512

In [190]:
tokenizer.model_max_length

512

In [191]:
model = Decoder(
    vocab_size=tokenizer.vocab_size,
    max_len=tokenizer.model_max_length,
    d_k=16,
    d_model=64,
    n_heads=4,
    n_layers=2,
    dropout_prob=0.1
)

model.to(device)

Decoder(
  (embedding): Embedding(28996, 64)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mha): CausalSelfAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
      )
      (ann): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=64, bias=True)
        (3): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (ln1): LayerNorm((64,), eps=1e-05,

In [192]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = torch.optim.Adam(model.parameters())

In [193]:
from datetime import datetime

In [194]:
def train(model, criterion, optimizer, train_loader, epochs):
  train_losses = np.zeros(epochs)

  for it in range(epochs):
    model.train()
    t0 = datetime.now()
    train_loss = []
    for batch in train_loader:
      # move data to GPU
      batch = {k: v.to(device) for k, v in batch.items()}

      # zero the parameter gradients
      optimizer.zero_grad()

      # shift targets backwards
      targets = batch['input_ids'].clone().detach()
      targets = torch.roll(targets, shifts=-1, dims=1)
      targets[:, -1] = tokenizer.pad_token_id

      # forward pass
      outputs = model(batch['input_ids'], batch['attention_mask'])
      # Ouputs are N x T x V
      # But PyTorch expect N x V x T

      loss = criterion(outputs.transpose(2, 1), targets)

      # backward and optimizer
      loss.backward()
      optimizer.step()
      train_loss.append(loss.item())

    # get average train loss
    train_loss = np.mean(train_loss)

    train_losses[it] = train_loss

    dt = datetime.now() - t0
    print(f'Epoch {it+1}/{epochs}, Train loss: {train_loss:.4f}, Duration: {dt}')

  return train_losses




In [195]:
train_losses = train(
    model, criterion, optimizer, train_loader, epochs=15
)

Epoch 1/15, Train loss: 5.9681, Duration: 0:01:07.848639
Epoch 2/15, Train loss: 5.0171, Duration: 0:01:07.704404
Epoch 3/15, Train loss: 4.6826, Duration: 0:01:00.100373
Epoch 4/15, Train loss: 4.4929, Duration: 0:01:00.137546
Epoch 5/15, Train loss: 4.3594, Duration: 0:01:00.678141
Epoch 6/15, Train loss: 4.2516, Duration: 0:01:00.337167
Epoch 7/15, Train loss: 4.1652, Duration: 0:01:00.369228
Epoch 8/15, Train loss: 4.0903, Duration: 0:01:00.293127
Epoch 9/15, Train loss: 4.0232, Duration: 0:01:00.132351
Epoch 10/15, Train loss: 3.9609, Duration: 0:00:59.812563
Epoch 11/15, Train loss: 3.9052, Duration: 0:01:00.401173
Epoch 12/15, Train loss: 3.8581, Duration: 0:00:59.715812
Epoch 13/15, Train loss: 3.8107, Duration: 0:00:59.922591
Epoch 14/15, Train loss: 3.7702, Duration: 0:01:00.457132
Epoch 15/15, Train loss: 3.7291, Duration: 0:01:00.195744


In [196]:
valid_loader = DataLoader(
    tokenized_datasets['validation'],
    batch_size=1,
    collate_fn=data_collator
)

In [237]:
model.eval()
for batch in valid_loader:
  # move data to GPU
  batch = {k: v.to(device) for k, v in batch.items()}

  # forward pass
  outputs = model(batch['input_ids'], batch['attention_mask'])

  break

In [238]:
outputs.shape

torch.Size([1, 12, 28996])

In [239]:
torch.argmax(outputs, axis=-1)

tensor([[ 170,  112,  188,  170, 1363, 1105, 8362, 6276,  102,  102,  102, 1107]],
       device='cuda:0')

In [240]:
prediction_ids = torch.argmax(outputs, axis=-1)

In [241]:
tokenizer.decode(prediction_ids[0])

"a ' s a good and un funny [SEP] [SEP] [SEP] in"

In [242]:
tokenizer.decode(batch['input_ids'][0])

"[CLS] it ' s a charming and often affecting journey. [SEP]"

In [243]:
tokenizer.decode(torch.concat((batch['input_ids'][0, :5], prediction_ids[:, 4])))

"[CLS] it ' s a good"

In [280]:
prompt = "it's"

In [281]:
tokenized_prompt = tokenizer(prompt, return_tensors='pt')
tokenized_prompt

{'input_ids': tensor([[ 101, 1122,  112,  188,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

In [283]:
tokenizer.decode(tokenized_prompt['input_ids'][:, :-1])

["[CLS] it ' s"]

In [277]:
outputs = model(
    tokenized_prompt['input_ids'][:, :-1].to(device),
    tokenized_prompt['attention_mask'][:, :-1].to(device)
)

outputs.shape

torch.Size([1, 4, 28996])

In [278]:
prediction_ids = torch.argmax(outputs[:, -1, :], axis=-1)

In [279]:
tokenizer.decode(prediction_ids[0])

'a'

In [262]:
prompt = "it's a"

In [263]:
tokenized_prompt = tokenizer(prompt, return_tensors='pt')

In [264]:
input_ids = tokenized_prompt['input_ids'][:, :-1].to(device)
mask = tokenized_prompt['attention_mask'][:, :-1].to(device)

In [265]:
for _ in range(20):
  outputs = model(input_ids, mask)
  prediction_id = torch.argmax(outputs[:, -1, :], axis=-1)
  input_ids = torch.hstack((input_ids, prediction_id.view(1, 1)))
  mask = torch.ones_like(input_ids)

  if prediction_id == tokenizer.sep_token_id:
    break

In [266]:
tokenizer.decode(input_ids[0])

"[CLS] it ' s a good time [SEP]"